# Clean Nepali Movies Data


In [1]:
from pathlib import Path
import json
import re
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "data cleaning.ipynb").exists():
    NOTEBOOK_DIR = Path("/Users/prabinthakur/code101/test_apps/nepali movies review/nlp projects/text preprocessing ").resolve()

RAW_PATH = NOTEBOOK_DIR.parent / "data acquisition" / "movies_raw.jsonl"
OUTPUT_DIR = NOTEBOOK_DIR


In [2]:
records = []
malformed = []

with RAW_PATH.open("r", encoding="utf-8") as file:
    for line_number, line in enumerate(file, 1):
        if not line.strip():
            continue
        try:
            record = json.loads(line)
        except json.JSONDecodeError as error:
            malformed.append((line_number, str(error)))
            continue
        if record.get("status", "ok") == "ok":
            records.append(record)

seen = set()
clean_records = []
for record in records:
    movie_id = record.get("id")
    slug = record.get("slug")
    key = ("id", str(movie_id).strip()) if movie_id not in (None, "") else ("slug", str(slug).strip() if slug else "")
    if key not in seen:
        seen.add(key)
        clean_records.append(record)

print(f"Parsed/status-ok: {len(records)}")
print(f"Malformed JSON lines: {len(malformed)}")
for line_number, error in malformed:
    print(f"  line {line_number}: {error}")
print(f"After deduplication: {len(clean_records)}")


Parsed/status-ok: 1803
Malformed JSON lines: 0
After deduplication: 1803


In [3]:
def text(value):
    return value.strip() if isinstance(value, str) else None


def string_list(value):
    if not isinstance(value, list):
        value = [value] if isinstance(value, str) else []
    return list(dict.fromkeys(item.strip() for item in value
                             if isinstance(item, str) and item.strip()))


def people(value):
    if isinstance(value, dict):
        value = [value]
    if not isinstance(value, list):
        return []
    return list(dict.fromkeys(text(person.get("name")) for person in value
                             if isinstance(person, dict) and text(person.get("name"))))


def nullable_int(value):
    if value is None or isinstance(value, bool):
        return pd.NA
    number = pd.to_numeric(value, errors="coerce")
    return pd.NA if pd.isna(number) else int(number)


def duration(value):
    if value is None or isinstance(value, bool):
        return pd.NA
    if isinstance(value, (int, float)):
        return pd.NA if pd.isna(value) else int(round(float(value)))

    value = str(value).strip()
    number = pd.to_numeric(value, errors="coerce")
    if not pd.isna(number):
        return int(round(float(number)))

    match = re.fullmatch(r"P(?:(\d+(?:\.\d+)?)D)?(?:T(?:(\d+(?:\.\d+)?)H)?(?:(\d+(?:\.\d+)?)M)?(?:(\d+(?:\.\d+)?)S)?)?", value, re.I)
    if not match:
        return pd.NA
    days, hours, minutes, seconds = (float(part) if part else 0 for part in match.groups())
    return int(round(days * 1440 + hours * 60 + minutes + seconds / 60))


def score(value):
    match = re.search(r"(-?\d+(?:\.\d+)?)", str(value)) if value is not None else None
    return float(match.group(1)) if match else pd.NA


movie_rows = []
review_rows = []

for record in clean_records:
    movie_id_value = record.get("id")
    slug = text(record.get("slug"))
    movie_id = str(movie_id_value).strip() if movie_id_value not in (None, "") else slug

    date_published = pd.to_datetime(record.get("datePublished"), errors="coerce", format="mixed")
    release_year = nullable_int(record.get("catalog_year"))
    if pd.isna(release_year) and not pd.isna(date_published):
        release_year = int(date_published.year)

    movie_rows.append({
        "movie_id": movie_id,
        "slug": slug,
        "title": text(record.get("title")) or text(record.get("ld_name")),
        "alternate_names": string_list(record.get("alternateName")),
        "description": text(record.get("description")),
        "genres": string_list(record.get("genre")),
        "date_published": date_published,
        "release_year": release_year,
        "duration_mins": duration(record.get("duration")),
        "directors": people(record.get("director")),
        "cast": people(record.get("actor")),
        "total_ratings": nullable_int(record.get("catalog_total_ratings")),
        "reviews_count": nullable_int(record.get("reviews_count")),
    })

    reviews = record.get("reviews")
    if not isinstance(reviews, list):
        reviews = []
    for review in reviews:
        if not isinstance(review, dict):
            continue
        review_text = text(review.get("text"))
        review_rows.append({
            "movie_id": movie_id,
            "slug": slug,
            "author": text(review.get("author")),
            "review_date": pd.to_datetime(review.get("date"), errors="coerce", format="mixed"),
            "score_10": score(review.get("score")),
            "text": review_text,
            "has_written_review": bool(review_text and review_text != "User shared a rating only."),
        })

movies_df = pd.DataFrame(movie_rows)
reviews_df = pd.DataFrame(review_rows)

movies_df["date_published"] = pd.to_datetime(movies_df["date_published"], errors="coerce")
for column in ("release_year", "duration_mins", "total_ratings", "reviews_count"):
    movies_df[column] = pd.to_numeric(movies_df[column], errors="coerce").astype("Int64")
reviews_df["review_date"] = pd.to_datetime(reviews_df["review_date"], errors="coerce")
reviews_df["score_10"] = pd.to_numeric(reviews_df["score_10"], errors="coerce")
reviews_df["has_written_review"] = reviews_df["has_written_review"].astype("bool")

print("movies_df:", movies_df.shape)
print("reviews_df:", reviews_df.shape)


movies_df: (1803, 13)
reviews_df: (389, 7)


In [4]:
print("=== Data Quality Audit ===")
print("movies_df shape:", movies_df.shape)
print("reviews_df shape:", reviews_df.shape)

for name, frame in (("movies_df", movies_df), ("reviews_df", reviews_df)):
    def empty_count(column):
        total = 0
        for value in column:
            if isinstance(value, (list, tuple)):
                total += len(value) == 0
            elif isinstance(value, str):
                total += value == ""
            else:
                total += bool(pd.isna(value))
        return total

    counts = pd.DataFrame({
        "null_count": frame.isna().sum(),
        "empty_count": frame.apply(empty_count),
    })
    print(f"\n{name} null/empty counts:")
    print(counts)

print("\nGenre frequencies:")
print(movies_df["genres"].explode().dropna().value_counts().to_string())

boilerplate = movies_df["description"].fillna("").str.contains("is a movie directed by", case=False, regex=False)
print("\nBoilerplate descriptions:", int(boilerplate.sum()))

review_counts = reviews_df["has_written_review"].value_counts().rename(index={True: "written", False: "rating-only"})
print("\nReview types:")
print(review_counts.to_string())

movies_csv = movies_df.copy()
for column in ("alternate_names", "genres", "directors", "cast"):
    movies_csv[column] = movies_csv[column].apply(lambda values: "   ".join(values))

movies_df.to_parquet(OUTPUT_DIR / "movies_clean.parquet", index=False)
reviews_df.to_parquet(OUTPUT_DIR / "reviews_clean.parquet", index=False)
movies_csv.to_csv(OUTPUT_DIR / "movies_clean.csv", index=False)

print("\nExported:")
print(OUTPUT_DIR / "movies_clean.parquet")
print(OUTPUT_DIR / "reviews_clean.parquet")
print(OUTPUT_DIR / "movies_clean.csv")


=== Data Quality Audit ===
movies_df shape: (1803, 13)
reviews_df shape: (389, 7)

movies_df null/empty counts:
                 null_count  empty_count
movie_id                  0            0
slug                      0            0
title                     0            0
alternate_names           0           86
description               0         1481
genres                    0            0
date_published            0            0
release_year              0            0
duration_mins            93           93
directors                 0           61
cast                      0            1
total_ratings             0            0
reviews_count             0            0

reviews_df null/empty counts:
                    null_count  empty_count
movie_id                     0            0
slug                         0            0
author                       0            0
review_date                  0            0
score_10                    20           20
text               

In [5]:
import os
import urllib.parse
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests

movies_df = pd.read_parquet(OUTPUT_DIR / "movies_clean.parquet")
if "description_source" not in movies_df:
    movies_df["description_source"] = pd.Series([pd.NA] * len(movies_df), index=movies_df.index, dtype="string")
if "description_url" not in movies_df:
    movies_df["description_url"] = pd.Series([pd.NA] * len(movies_df), index=movies_df.index, dtype="string")

original_mask = movies_df["description"].notna() & movies_df["description"].astype("string").str.strip().ne("")
unclassified_original = original_mask & movies_df["description_source"].isna()
movies_df.loc[unclassified_original, "description_source"] = "original"

checkpoint_path = OUTPUT_DIR / "movies_enriched_checkpoint.jsonl"
checkpoint = {}
if checkpoint_path.exists():
    with checkpoint_path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                row = json.loads(line)
                checkpoint[str(row["movie_id"])] = row

missing_mask = movies_df["description"].isna() | movies_df["description"].astype("string").str.strip().eq("")
priority_mask = missing_mask & (
    (movies_df["release_year"].fillna(0) >= 2020) |
    (movies_df["total_ratings"].fillna(0) > 0)
)
movie_ids = movies_df["movie_id"].astype("string")
priority_ids = movies_df.loc[priority_mask & ~movie_ids.isin(checkpoint), "movie_id"].tolist()
earlier_ids = movies_df.loc[missing_mask & ~priority_mask & ~movie_ids.isin(checkpoint), "movie_id"].tolist()
ordered_movie_ids = priority_ids + earlier_ids

print(f"Missing descriptions: {int(missing_mask.sum())}")
print(f"Priority remaining: {len(priority_ids)}")
print(f"Earlier-year remaining: {len(earlier_ids)}")
print(f"Checkpoint entries: {len(checkpoint)}")

session = requests.Session()
session.headers.update({"User-Agent": "Nepali-Movies-Data-Cleaning/1.0"})


def norm(value):
    value = "" if value is None else str(value)
    return re.sub(r"[^a-z0-9]+", " ", value.lower()).strip()


def clean_synopsis(value):
    if not isinstance(value, str):
        return None
    lines = []
    for line in value.replace("\r", "\n").split("\n"):
        line = line.strip()
        if not line:
            continue
        lowered = line.lower()
        if re.match(r"^(subscribe|cast & crew|genre|director|starring|released by|official trailer)", lowered):
            continue
        if lowered.startswith("for more") or lowered.startswith("watch full"):
            continue
        lines.append(line)
    result = re.sub(r"\s+", " ", " ".join(lines)).strip()
    return result if len(result) >= 40 else None


def title_matches(row, candidate_titles):
    accepted = {norm(row["title"])}
    accepted.update(norm(name) for name in row["alternate_names"] if isinstance(name, str))
    return any(norm(title) in accepted or norm(title) == norm(row["title"]) for title in candidate_titles if title)


def year_matches(candidate_year, expected_year):
    if pd.isna(expected_year):
        return True
    try:
        return abs(int(candidate_year) - int(expected_year)) <= 1
    except (TypeError, ValueError):
        return True


def director_matches(row, text_value):
    directors = row.get("directors") if isinstance(row.get("directors"), list) else []
    if not directors:
        return True
    haystack = norm(text_value)
    matches = [norm(director) in haystack for director in directors if norm(director)]
    return not matches or any(matches)


def tmdb_lookup(row):
    api_key = os.getenv("TMDB_API_KEY")
    if not api_key:
        return None
    names = [row["title"]] + (row["alternate_names"] if isinstance(row["alternate_names"], list) else [])
    for name in names:
        if not name:
            continue
        try:
            response = session.get(
                "https://api.themoviedb.org/3/search/movie",
                params={"api_key": api_key, "query": name, "include_adult": "false"},
                timeout=15,
            )
            response.raise_for_status()
            for candidate in response.json().get("results", [])[:5]:
                if not title_matches(row, [candidate.get("title"), candidate.get("original_title")]):
                    continue
                if not year_matches(candidate.get("release_date", "")[:4], row["release_year"]):
                    continue
                overview = clean_synopsis(candidate.get("overview"))
                if overview:
                    movie_id = candidate["id"]
                    return overview, "tmdb", f"https://www.themoviedb.org/movie/{movie_id}"
        except requests.RequestException:
            continue
    return None


def wikipedia_lookup(row):
    names = [row["title"]] + (row["alternate_names"] if isinstance(row["alternate_names"], list) else [])
    for name in names:
        if not name:
            continue
        try:
            response = session.get(
                "https://en.wikipedia.org/w/api.php",
                params={"action": "query", "list": "search", "srsearch": f'"{name}" Nepali film', "srlimit": 5, "format": "json"},
                timeout=15,
            )
            response.raise_for_status()
            candidates = response.json().get("query", {}).get("search", [])
            for candidate in candidates:
                page_title = candidate.get("title", "")
                if not title_matches(row, [page_title]):
                    continue
                page_response = session.get(
                    "https://en.wikipedia.org/w/api.php",
                    params={"action": "query", "prop": "extracts|pageprops", "exintro": 1, "explaintext": 1, "titles": page_title, "format": "json", "formatversion": 2},
                    timeout=15,
                )
                page_response.raise_for_status()
                pages = page_response.json().get("query", {}).get("pages", [])
                if not pages:
                    continue
                extract = clean_synopsis(pages[0].get("extract", ""))
                if not extract:
                    continue
                year_tokens = re.findall(r"\b(?:19|20)\d{2}\b", extract)
                if year_tokens and not pd.isna(row["release_year"]):
                    if not any(year_matches(year, row["release_year"]) for year in year_tokens):
                        continue
                if not director_matches(row, extract):
                    continue
                url = "https://en.wikipedia.org/wiki/" + urllib.parse.quote(page_title.replace(" ", "_"))
                return extract, "wikipedia", url
        except requests.RequestException:
            continue
    return None


def imdb_lookup(row):
    names = [row["title"]] + (row["alternate_names"] if isinstance(row["alternate_names"], list) else [])
    for name in names:
        if not name:
            continue
        slug = urllib.parse.quote(str(name).lower().replace(" ", "-"))
        prefix = slug[0] if slug else "a"
        try:
            response = session.get(f"https://v2.sg.media-imdb.com/suggestion/{prefix}/{slug}.json", timeout=15)
            response.raise_for_status()
            candidates = response.json().get("d", [])
            for candidate in candidates[:8]:
                if not title_matches(row, [candidate.get("l")]):
                    continue
                if not year_matches(candidate.get("y"), row["release_year"]):
                    continue
                title_id = candidate.get("id")
                if not title_id:
                    continue
                page = session.get(f"https://www.imdb.com/title/{title_id}/", timeout=15)
                page.raise_for_status()
                match = re.search(r'<script type="application/ld\\+json">(.*?)</script>', page.text, re.S)
                if not match:
                    continue
                data = json.loads(match.group(1))
                directors = data.get("director", [])
                if isinstance(directors, dict):
                    directors = [directors]
                director_names = [director.get("name") for director in directors if isinstance(director, dict)]
                if row["directors"] and director_names:
                    expected = {norm(name) for name in row["directors"]}
                    found = {norm(name) for name in director_names}
                    if expected.isdisjoint(found):
                        continue
                description = clean_synopsis(data.get("description"))
                if description:
                    return description, "imdb", f"https://www.imdb.com/title/{title_id}/"
        except (requests.RequestException, json.JSONDecodeError):
            continue
    return None


def enrich_row(movie_id):
    mask = movies_df["movie_id"].astype("string") == str(movie_id)
    row = movies_df.loc[mask].iloc[0].to_dict()
    result = tmdb_lookup(row)
    if result is None:
        result = wikipedia_lookup(row)
    if result is None:
        result = imdb_lookup(row)

    return {
        "movie_id": str(row["movie_id"]),
        "slug": row.get("slug"),
        "title": row.get("title"),
        "release_year": None if pd.isna(row.get("release_year")) else int(row["release_year"]),
        "description": result[0] if result else None,
        "description_source": result[1] if result else None,
        "description_url": result[2] if result else None,
    }


batch_size = 25
processed = 0
for start in range(0, len(ordered_movie_ids), batch_size):
    batch = ordered_movie_ids[start:start + batch_size]
    batch_results = []
    with ThreadPoolExecutor(max_workers=4) as executor:
        futures = {executor.submit(enrich_row, movie_id): movie_id for movie_id in batch}
        for future in as_completed(futures):
            try:
                batch_results.append(future.result())
            except Exception:
                movie_id = futures[future]
                batch_results.append({"movie_id": str(movie_id), "description": None, "description_source": None, "description_url": None})

    with checkpoint_path.open("a", encoding="utf-8") as handle:
        for result in batch_results:
            handle.write(json.dumps(result, ensure_ascii=False) + "\n")

    for result in batch_results:
        mask = movies_df["movie_id"].astype("string") == result["movie_id"]
        if result["description"]:
            movies_df.loc[mask, "description"] = result["description"]
            movies_df.loc[mask, "description_source"] = result["description_source"]
            movies_df.loc[mask, "description_url"] = result["description_url"]

    processed += len(batch_results)
    print(f"Checkpointed batch: {processed}/{len(ordered_movie_ids)}")

for movie_id, result in checkpoint.items():
    if result.get("description"):
        mask = movies_df["movie_id"].astype("string") == str(movie_id)
        movies_df.loc[mask, "description"] = result["description"]
        movies_df.loc[mask, "description_source"] = result.get("description_source")
        movies_df.loc[mask, "description_url"] = result.get("description_url")

still_missing = movies_df["description"].isna() | movies_df["description"].astype("string").str.strip().eq("")
movies_df.loc[still_missing, "description"] = pd.NA
movies_df.loc[still_missing, "description_source"] = pd.NA
movies_df.loc[still_missing, "description_url"] = pd.NA

movies_df.to_parquet(OUTPUT_DIR / "movies_clean.parquet", index=False)

movies_csv = movies_df.copy()
for column in ("alternate_names", "genres", "directors", "cast"):
    movies_csv[column] = movies_csv[column].apply(lambda values: "   ".join(values) if isinstance(values, list) else "")
movies_csv.to_csv(OUTPUT_DIR / "movies_clean.csv", index=False)

print("Enrichment complete")
newly_filled = movies_df["description"].notna() & ~original_mask
print("Newly filled:", int(newly_filled.sum()))
print("Still missing:", int(still_missing.sum()))

filled_mask = movies_df["description_source"].notna() & movies_df["description_source"].ne("original")
summary = movies_df.loc[filled_mask].copy()
summary["decade"] = (summary["release_year"].fillna(0) // 10 * 10).astype("Int64")
print("\nFilled counts by description_source:")
print(summary.groupby("description_source", dropna=False).size().to_string())
print("\nFilled counts by release_year decade:")
print(summary.groupby("decade", dropna=False).size().to_string())


Missing descriptions: 1481
Priority remaining: 233
Earlier-year remaining: 1248
Checkpoint entries: 0


Checkpointed batch: 25/1481


Checkpointed batch: 50/1481


Checkpointed batch: 75/1481


Checkpointed batch: 100/1481


Checkpointed batch: 125/1481


... (51 outputs left out)

Checkpointed batch: 1425/1481


Checkpointed batch: 1450/1481


Checkpointed batch: 1475/1481


Checkpointed batch: 1481/1481
Enrichment complete
Newly filled: 0
Still missing: 1481

Filled counts by description_source:
Series([], )

Filled counts by release_year decade:
Series([], )
